In [1]:
# GoogLeNet — Per-channel Symmetric PTQ (FX, aux heads on)

import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import googlenet
from sklearn.model_selection import train_test_split

# FX Quantization imports 
from torch.ao.quantization import QConfig, QConfigMapping
from torch.ao.quantization.observer import MinMaxObserver, PerChannelMinMaxObserver
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

# Repro & backend
torch.backends.quantized.engine = "fbgemm"
torch.manual_seed(42); np.random.seed(42)
device = torch.device("cpu")  # PTQ/eval on CPU


# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"   #  change to your path
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")

targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(
        idxs, test_size=0.15, random_state=42, shuffle=True
    )
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=1/3, random_state=42, shuffle=True
    )
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size  = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")


# Load trained FP32 GoogLeNet 

ckpt_path = "/home/h5/sapi279d/googlenet_weights.pth"  # checkpoint
state_dict = torch.load(ckpt_path, map_location="cpu")
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

NUM_CLASSES = len(dataset.classes)  # e.g., 200

model_fp32 = googlenet(weights=None, aux_logits=True)  # keep aux1/aux2
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, NUM_CLASSES)
model_fp32.aux1.fc2 = nn.Linear(model_fp32.aux1.fc2.in_features, NUM_CLASSES)
model_fp32.aux2.fc2 = nn.Linear(model_fp32.aux2.fc2.in_features, NUM_CLASSES)

model_fp32.load_state_dict(state_dict, strict=True)
model_fp32.eval()
print("FP32 GoogLeNet loaded")


# Per-channel Symmetric PTQ (FX)

per_channel_qconfig = QConfig(
    activation=MinMaxObserver.with_args(
        dtype=torch.quint8, qscheme=torch.per_tensor_affine
    ),
    weight=PerChannelMinMaxObserver.with_args(
        dtype=torch.qint8, qscheme=torch.per_channel_symmetric
    ),
)
qconfig_mapping = QConfigMapping().set_global(per_channel_qconfig)
example_inputs = (torch.randn(1, 3, 224, 224),)

prepared = prepare_fx(copy.deepcopy(model_fp32), qconfig_mapping, example_inputs)

# Calibration: feed a few batches
def calibrate_fx(model, loader, num_batches=20):
    model.eval()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            out = model(images)
            # prepared model in eval should return logits; if tuple, take first
            if isinstance(out, tuple):
                _ = out[0]

calibrate_fx(prepared, val_loader, num_batches=20)
print("Calibration done")

# Convert to INT8
model_int8 = convert_fx(prepared).eval()
print("Converted to per-channel symmetric INT8")


# Evaluation

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            if isinstance(outputs, tuple):  # safety for any tuple returns
                outputs = outputs[0]
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_int8 = evaluate_model(model_int8, test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy (Per-channel Sym, FX): {acc_int8:.2f}%")


# Model size & Runtime

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    ms_per_batch = (total_time / num_batches) * 1000
    ms_per_image = (total_time / total_images) * 1000
    return ms_per_batch, ms_per_image

fp32_size = get_model_size(model_fp32)
int8_size = get_model_size(model_int8)
print(f"FP32 size: {fp32_size:.2f} MB | INT8 size: {int8_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_int8, img_ms_int8 = benchmark(model_int8, test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"INT8 Inference: {batch_ms_int8:.2f} ms/batch | {img_ms_int8:.2f} ms/image")


Total images: 100000, Classes: 200
DataLoaders ready


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


FP32 GoogLeNet loaded
Calibration done
Converted to per-channel symmetric INT8
FP32 Accuracy: 45.10%
INT8 Accuracy (Per-channel Sym, FX): 43.08%
FP32 size: 42.34 MB | INT8 size: 6.03 MB
FP32 Inference: 2509.55 ms/batch | 19.61 ms/image
INT8 Inference: 1097.06 ms/batch | 8.57 ms/image
